In [ ]:
import time
import os

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# from brax import envs

In [ ]:
import jax
import flax.linen as nn
import jax.numpy as jnp
# from wrappers import AutoResetWrapper
from brax import envs
# from flax import serialization

In [ ]:
from custom_types import RNGKey, EnvState, Params
# from utils import vx_transition_fn

In [ ]:
from typing import Any, Tuple, List
from algorithms import TrajectoryPPO, PPOConfigs, PPOTrainingState
from buffer import PPOTransition
# from networks import MLP, PPO_Policy
from networks import GCMLP, GC_PPO_Policy
from functools import partial
from trajectory_encoder import TaskState, TrajectoryEncoder, CircularEncoder, LineEncoder

In [ ]:
algo_name = "PPO"

In [ ]:
critic_hidden_layers: Tuple[int, ...] = (64, 64)
actor_hidden_layers: Tuple[int, ...] = (64, 64)
vec_env = 2048

ppo_config = PPOConfigs(
    policy_learnng_rate=1e-4,
    critic_learning_rate=1e-4,
    clip_ratio=0.2,
    entropy_gain=0.005,
    discount=0.99,
    td_lambda_discount=0.95,
    rollout_length=128,
    vec_env=vec_env,
    mini_batch_size=512,
    critic_epochs=1,
    policy_epochs=1,
    initial_std=0.25,
    std_decay_rate=0.0001,
    min_std=0.1,
)

In [ ]:
# set random seed
# seed = 42
seed = 213
loop_random_key = jax.random.PRNGKey(seed)
loop_random_key, subkey = jax.random.split(loop_random_key)

# # creat environment (Ant)
env = envs.create(env_name="ant", episode_length=4096, backend="mjx", auto_reset=True)
# env = AutoResetWrapper(env) # custom auto reset

# env = ChaserEvaderEnv()
# env_params = env.default_params

# define networks
policy_network = GC_PPO_Policy(
    hidden_layer_sizes=actor_hidden_layers,
    action_dim=env.action_size,
    initial_std=0.1 * jnp.ones(env.action_size),
    kernel_init=jax.nn.initializers.orthogonal(jnp.sqrt(2)),
    kernel_init_final=jax.nn.initializers.orthogonal(0.01),
    activation=nn.softplus,
    # activation=nn.relu,
    final_activation=jnp.tanh,
    learnable_std=False,
    # learnable_std=True,
)

critic_network = GCMLP(
    layer_sizes=critic_hidden_layers + (1,),
    kernel_init=jax.nn.initializers.orthogonal(jnp.sqrt(2)),
    activation=nn.softplus,
    kernel_init_final=jax.nn.initializers.orthogonal(0.01),
    final_activation=lambda x: x - 3,
)

In [ ]:
# task_encoder = TrajectoryEncoder(var=0.25, l=2)
task_encoder = CircularEncoder(v_max=2.5)
# task_encoder = LineEncoder(v_max=3.0)

ppo = TrajectoryPPO(
    env=env,
    policy_network=policy_network,
    critic_network=critic_network,
    ppo_configs=ppo_config,
    encoder=task_encoder,
)

ppo_training_state = ppo.init(subkey)

In [ ]:
seed = 42
loop_random_key = jax.random.PRNGKey(seed)
loop_random_key, subkey = jax.random.split(loop_random_key)
subkeys = jax.random.split(subkey, num=vec_env)
states = jax.vmap(env.reset)(subkeys)

# loop_random_key, subkey = jax.random.split(loop_random_key)
# subkeys = jax.random.split(subkey, num=vec_env)
# backup_states = jax.vmap(env.reset)(subkeys)

loop_random_key, subkey = jax.random.split(loop_random_key)
subkeys = jax.random.split(subkey, num=vec_env)
task_states = jax.vmap(task_encoder.reset)(states.obs, subkeys)
# last_actions = jnp.zeros((vec_env, env.action_size))


In [ ]:
def training_loop(
    carry: Tuple[EnvState, TaskState, PPOTrainingState, RNGKey], 
    _: None,
    ) -> Tuple[Tuple, Tuple]:

    states, task_states, ppo_training_state, loop_random_key = carry

    (states, task_states, ppo_training_state, loop_random_key), transitions = ppo.train(
        states,
        task_states,
        ppo_training_state,
        loop_random_key,
    )

    vs = jnp.sqrt(jnp.sum(task_states.task[:, :2]**2, axis=-1))

    initial_states = states.replace(
        pipeline_state=states.info['first_pipeline_state'], 
        obs=states.info['first_obs'],
        )

    loop_random_key, subkey = jax.random.split(loop_random_key)
    ps = jax.random.bernoulli(subkey, 0.125, shape=(vec_env,))
    states = jax.tree.map(
        lambda a, b: jax.vmap(jax.lax.select)(ps, a, b),
        initial_states,
        states,
    )

    loop_random_key, subkey = jax.random.split(loop_random_key)
    subkeys = jax.random.split(subkey, num=vec_env)
    candidate_task_states = jax.vmap(task_encoder.reset)(states.obs, subkeys)
    
    loop_random_key, subkey = jax.random.split(loop_random_key)
    ps = ps | jax.random.bernoulli(subkey, 1/7, shape=(vec_env,))
    task_states = jax.tree.map(
        lambda a, b: jax.vmap(jax.lax.select)(ps, a, b),
        candidate_task_states,
        task_states,
    )

    new_carry = (
        states,
        task_states,
        ppo_training_state,
        loop_random_key,
    )

    return new_carry, (jnp.mean(transitions.rewards), jnp.mean(transitions.td_lambda_returns), jnp.mean(vs))

In [ ]:
carry = (states, task_states, ppo_training_state, loop_random_key)

In [ ]:
num_iterations = 2000

iteration_mean_returns = []
iteration_mean_rewards = []
iteration_mean_vs = []

for i in range(int(num_iterations / 500)):

    (
        states, 
        task_states, 
        ppo_training_state, 
        loop_random_key,
        ), (
            iteration_mean_reward, 
            iteration_mean_return,
            iteration_mean_v,
            ) = jax.lax.scan(
        training_loop,
        carry,
        length=500,
    )

    loop_random_key, subkey = jax.random.split(loop_random_key)
    subkeys = jax.random.split(subkey, num=vec_env)
    states = jax.vmap(env.reset)(subkeys)

    loop_random_key, subkey = jax.random.split(loop_random_key)
    subkeys = jax.random.split(subkey, num=vec_env)
    task_states = jax.vmap(task_encoder.reset)(states.obs, subkeys)

    carry = (states, task_states, ppo_training_state, loop_random_key)

    iteration_mean_returns.append(iteration_mean_return)
    iteration_mean_rewards.append(iteration_mean_reward)
    iteration_mean_vs.append(iteration_mean_v)

    print(int((i + 1) * 50000 / num_iterations), "% complete")

states, task_states, ppo_training_state, loop_random_key = carry

(
    states, 
    task_states,
    ppo_training_state,
    loop_random_key
), transitions = ppo.train(
    states,
    task_states,
    ppo_training_state,
    loop_random_key,
)

carry = (states, task_states, ppo_training_state, loop_random_key)

In [ ]:
(
    states, 
    task_states,
    ppo_training_state,
    loop_random_key
), transitions = ppo.train(
    states,
    task_states,
    ppo_training_state,
    loop_random_key,
)

In [ ]:
iteration_mean_returns = jnp.concatenate(iteration_mean_returns)
iteration_mean_rewards = jnp.concatenate(iteration_mean_rewards)
iteration_mean_vs = jnp.concatenate(iteration_mean_vs)

In [ ]:
print(ppo_training_state.current_std)

if policy_network.learnable_std:
    print(nn.sigmoid(ppo_training_state.policy_params['params']['std_logits']))

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
plt.plot(jnp.arange(num_iterations), iteration_mean_returns)
plt.show()


In [ ]:
plt.plot(jnp.arange(num_iterations), iteration_mean_vs)
plt.show()

In [ ]:
# bytes_data = serialization.to_bytes(ppo_training_state.policy_params)
# filename = "xy_ant_policy.msgpack"

# with open(filename, "wb") as f:
#     f.write(bytes_data)
# print(f"Parameters saved to {filename}")

In [ ]:
print(jnp.mean(states.reward))

In [ ]:
print(jnp.mean(transitions.dones))

In [ ]:
print(transitions.dones.shape)
print(vec_env)

In [ ]:
vs = jnp.sqrt(jnp.sum(task_states.task[:, :2]**2, axis=-1))

In [ ]:
index = 34
rollout_length = 128
plt.plot(jnp.arange(rollout_length), transitions.rewards[:, index])
plt.plot(jnp.arange(rollout_length), transitions.dones[:, index])
plt.plot(jnp.arange(rollout_length), transitions.td_lambda_returns[:, index])

# plt.plot(jnp.arange(rollout_length), jnp.sqrt(jnp.sum(transitions.tasks[:, index, -2:]**2, axis=-1)))
plt.show()

print("v:", vs[index])
# print("w:", task_states.task[index][-1] * 57.3)

In [ ]:
jnp.sqrt((states.metrics["x_velocity"])**2 + (states.metrics["y_velocity"])**2)[index]

In [ ]:
print("average v:", jnp.mean(vs))

In [ ]:
task_states.deviation[index]

In [ ]:
# _task_states = jax.vmap(task_encoder.reset)(states.obs, subkeys)
# vs = jnp.sqrt(jnp.sum(_task_states.task[:, :2]**2, axis=-1))

In [ ]:
plt.hist(vs)
plt.show()
# vs.shape

In [ ]:
# from flax import serialization

In [ ]:
# bytes_output = serialization.to_bytes(ppo_training_state.policy_params)

# # 2. 写入文件
# with open("model.msgpack", "wb") as f:
#     f.write(bytes_output)